# 🥇 Capa Gold: Detección de Lavado de Dinero (AML) con Machine Learning y Delta Lake

Este notebook procesa los datos limpios de la capa **Silver** (`transacciones_plata_robert` y `Dim_Cliente`), implementa ingeniería de características (Feature Engineering) distribuida con **PySpark**, aplica algoritmos de detección de anomalías y reglas normativas antilavado (GAFI / FATF), y genera las tablas maestras de consumo analítico en formato **Delta Lake**:

1. **`gold_perfiles_riesgo_cliente`**: Visión 360 del cliente con scores de anomalía, nivel de riesgo y tipologías detectadas.
2. **`gold_alertas_aml`**: Detalle transaccional de alertas para analistas de cumplimiento (Reportes de Operaciones Sospechosas - ROS).

In [0]:
# 1. Importación de Librerías y Configuración
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print("✅ Librerías importadas correctamente.")

In [0]:
# 2. Lectura de Datos desde la Capa Silver (Delta Lake y Azure SQL)
df_transacciones_silver = spark.table("aml_proyect.transacciones_plata_robert")

# Lectura de Dim_Cliente desde Azure SQL o tabla local
jdbc_password = dbutils.secrets.get(scope="azure_secret", key="sql_pwd")
jdbc_url = "jdbc:sqlserver://sqldb-pitufeotracking-source.database.windows.net:1433;database=free-sql-db-9553025;encrypt=true;trustServerCertificate=True;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

df_clientes_silver = (spark.read
  .format("jdbc")
  .option("url", jdbc_url)
  .option("dbtable", "Dim_Cliente")
  .option("user", "admin_robert@sqldb-pitufeotracking-source")
  .option("password", jdbc_password)
  .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
  .load()
)

print(f"Transacciones Silver: {df_transacciones_silver.count()} registros")
print(f"Clientes Silver: {df_clientes_silver.count()} registros")

In [0]:
# 3. Feature Engineering Distribuido (PySpark)
print("🛠️ Calculando métricas agregadas por cliente...")

# Agregaciones de transacciones de salida (Origen)
agg_origen = df_transacciones_silver.groupBy("FK_Cliente_Origen").agg(
    F.count("PK_Transaccion").alias("Total_Tx_Enviadas"),
    F.sum("Monto_USD").alias("Monto_Total_Enviado_USD"),
    F.avg("Monto_USD").alias("Monto_Promedio_Enviado_USD"),
    F.max("Monto_USD").alias("Monto_Max_Enviado_USD"),
    F.stddev("Monto_USD").alias("Monto_Std_Enviado_USD"),
    F.countDistinct("FK_Cliente_Destino").alias("Destinatarios_Unicos"),
    F.sum(F.when(F.col("FK_Canal").rlike("(?i)Efectivo|ATM|Cajero|Ventanilla"), 1).otherwise(0)).alias("Tx_Canal_Efectivo_ATM"),
    F.sum(F.when(F.col("FK_Canal").rlike("(?i)Crypto|Exchange|Bridge"), 1).otherwise(0)).alias("Tx_Canal_Crypto"),
    F.sum(F.when(F.col("FK_Canal").rlike("(?i)Plataforma|Terceros"), 1).otherwise(0)).alias("Tx_Canal_Plataformas")
).withColumnRenamed("FK_Cliente_Origen", "PK_Cliente")

# Agregaciones de transacciones de entrada (Destino)
agg_destino = df_transacciones_silver.groupBy("FK_Cliente_Destino").agg(
    F.count("PK_Transaccion").alias("Total_Tx_Recibidas"),
    F.sum("Monto_USD").alias("Monto_Total_Recibido_USD"),
    F.avg("Monto_USD").alias("Monto_Promedio_Recibido_USD"),
    F.max("Monto_USD").alias("Monto_Max_Recibido_USD"),
    F.countDistinct("FK_Cliente_Origen").alias("Remitentes_Unicos")
).withColumnRenamed("FK_Cliente_Destino", "PK_Cliente")

# Unir con la dimensión de clientes
df_features = (df_clientes_silver
    .join(agg_origen, on="PK_Cliente", how="left")
    .join(agg_destino, on="PK_Cliente", how="left")
    .fillna(0)
)

# Cálculo de ratios avanzados
df_features = df_features.withColumn(
    "Total_Movido", F.col("Monto_Total_Enviado_USD") + F.col("Monto_Total_Recibido_USD")
).withColumn(
    "Ratio_Salida_vs_Entrada",
    F.when(F.col("Monto_Total_Recibido_USD") > 0, F.round(F.col("Monto_Total_Enviado_USD") / F.col("Monto_Total_Recibido_USD"), 4))
     .otherwise(F.when(F.col("Monto_Total_Enviado_USD") > 0, 999.0).otherwise(0.0))
).withColumn(
    "Ratio_Operado_vs_Ingreso_Declarado",
    F.round(F.col("Total_Movido") / (F.when(F.col("Ingreso_Mensual_Aprox") <= 0, 1000.0).otherwise(F.col("Ingreso_Mensual_Aprox")) * 12.0), 4)
).withColumn(
    "Indice_Fan_In",
    F.when(F.col("Total_Tx_Recibidas") > 0, F.round(F.col("Remitentes_Unicos") / F.col("Total_Tx_Recibidas"), 4))
     .otherwise(0.0)
).withColumn(
    "Ratio_Tx_Crypto",
    F.when(F.col("Total_Tx_Enviadas") > 0, F.round(F.col("Tx_Canal_Crypto") / F.col("Total_Tx_Enviadas"), 4))
     .otherwise(0.0)
)

display(df_features.limit(5))

In [0]:
# 4. Detección de Anomalías con Machine Learning (Isolation Forest) y Reglas AML
print("🤖 Entrenando modelo de Machine Learning y evaluando tipologías...")

# Convertir a Pandas para inferencia con Scikit-Learn
pdf_features = df_features.toPandas()

feature_cols = [
    "Total_Tx_Enviadas", "Monto_Total_Enviado_USD", "Monto_Promedio_Enviado_USD",
    "Monto_Max_Enviado_USD", "Monto_Std_Enviado_USD", "Destinatarios_Unicos",
    "Tx_Canal_Efectivo_ATM", "Tx_Canal_Crypto", "Tx_Canal_Plataformas", "Total_Tx_Recibidas",
    "Monto_Total_Recibido_USD", "Remitentes_Unicos", "Ratio_Salida_vs_Entrada",
    "Ratio_Operado_vs_Ingreso_Declarado", "Indice_Fan_In", "Ratio_Tx_Crypto"
]

x_data = pdf_features[feature_cols].replace([np.inf, -np.inf], 999.0).fillna(0)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_data)

iso_forest = IsolationForest(contamination=0.15, random_state=42, n_estimators=150)
iso_forest.fit(x_scaled)

decision_scores = iso_forest.decision_function(x_scaled)
min_s, max_s = decision_scores.min(), decision_scores.max()
ml_anomaly_score = (1 - (decision_scores - min_s) / (max_s - min_s)) * 100 if max_s > min_s else np.zeros(len(decision_scores))

pdf_features["ML_Anomaly_Score"] = np.round(ml_anomaly_score, 2)
pdf_features["ML_Is_Outlier"] = iso_forest.predict(x_scaled) == -1

# Reglas Heurísticas de Tipologías AML
cond_structuring = (
    (pdf_features["Total_Tx_Enviadas"].between(3, 10)) &
    (pdf_features["Monto_Max_Enviado_USD"] <= 3000.0) &
    (pdf_features["Monto_Total_Enviado_USD"].between(8000.0, 10500.0))
)

cond_fan_in = (
    (pdf_features["Total_Tx_Recibidas"] >= 40) &
    (pdf_features["Indice_Fan_In"] >= 0.70) &
    (pdf_features["Monto_Total_Enviado_USD"] >= 0.70 * pdf_features["Monto_Total_Recibido_USD"]) &
    (pdf_features["Monto_Total_Enviado_USD"] < 10000.0)
)

cond_crypto = (
    (pdf_features["Tx_Canal_Crypto"] >= 3) &
    (pdf_features["Monto_Max_Enviado_USD"] <= 2600.0) &
    (pdf_features["Ratio_Tx_Crypto"] >= 0.50)
)

cond_kyc = (
    (pdf_features["Ratio_Operado_vs_Ingreso_Declarado"] >= 3.0) &
    (pdf_features["Monto_Total_Enviado_USD"] > 5000.0)
)

pdf_features["Flag_Structuring_Pitufeo"] = cond_structuring.astype(int)
pdf_features["Flag_Fan_In_Plataforma"] = cond_fan_in.astype(int)
pdf_features["Flag_Crypto_Mixer"] = cond_crypto.astype(int)
pdf_features["Flag_Desvio_KYC"] = cond_kyc.astype(int)

rule_score = (
    pdf_features["Flag_Structuring_Pitufeo"] * 40.0 +
    pdf_features["Flag_Fan_In_Plataforma"] * 35.0 +
    pdf_features["Flag_Crypto_Mixer"] * 35.0 +
    pdf_features["Flag_Desvio_KYC"] * 25.0
).clip(upper=100.0)

pdf_features["Rule_Risk_Score"] = np.round(rule_score, 2)
pdf_features["AML_Risk_Score"] = np.round((0.50 * pdf_features["ML_Anomaly_Score"]) + (0.50 * pdf_features["Rule_Risk_Score"]), 2)

condiciones_nivel = [
    pdf_features["AML_Risk_Score"] >= 70.0,
    pdf_features["AML_Risk_Score"].between(40.0, 69.99),
    pdf_features["AML_Risk_Score"] < 40.0
]
etiquetas_nivel = ["ALTO RIESGO - ALERTA ROS", "MEDIO RIESGO - MONITOREO", "BAJO RIESGO - NORMAL"]
pdf_features["Nivel_Riesgo_AML"] = np.select(condiciones_nivel, etiquetas_nivel, default="BAJO RIESGO - NORMAL")

# Reconvertir a Spark DataFrame
df_gold_clientes = spark.createDataFrame(pdf_features)
display(df_gold_clientes.filter(F.col("AML_Risk_Score") >= 40.0))

In [0]:
# 5. Generación de Tabla de Alertas Transaccionales Gold
print("🚨 Creando vista de transacciones alertadas...")

df_clientes_alertados = df_gold_clientes.filter(F.col("AML_Risk_Score") >= 40.0).select(
    "PK_Cliente", "AML_Risk_Score", "Nivel_Riesgo_AML", "ML_Anomaly_Score", "Rule_Risk_Score",
    "Flag_Structuring_Pitufeo", "Flag_Fan_In_Plataforma", "Flag_Crypto_Mixer", "Flag_Desvio_KYC"
)

df_gold_alertas = (df_transacciones_silver
    .join(df_clientes_alertados, df_transacciones_silver.FK_Cliente_Origen == df_clientes_alertados.PK_Cliente, how="inner")
    .join(df_clientes_silver.select("PK_Cliente", "Edad", "Ocupacion_Declarada", "Ingreso_Mensual_Aprox"), 
          df_transacciones_silver.FK_Cliente_Origen == df_clientes_silver.PK_Cliente, how="left")
    .drop(df_clientes_silver.PK_Cliente)
)

display(df_gold_alertas.limit(10))

In [0]:
# 6. Persistencia y MERGE en Delta Lake (Capa Gold)
print("💾 Guardando tablas en formato Delta Lake...")

# Guardar Tabla Gold de Perfiles de Riesgo
(df_gold_clientes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("aml_proyect.gold_perfiles_riesgo_cliente")
)

# Guardar Tabla Gold de Alertas Transaccionales
(df_gold_alertas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("aml_proyect.gold_alertas_aml")
)

print("🎉 Tablas Gold guardadas exitosamente en el catálogo Delta Lake de Databricks!")